In [16]:
import pandas as pd
import numpy as np

import sys
sys.path.append(".")
from scoring import *

Load Data

In [17]:
df_player_fixtures = pd.read_csv("data/processed/player_fixtures.csv")
df_squads_rated = pd.read_csv("data/processed/squads_rated.csv")
df_fixtures = pd.read_csv("data/processed/fixtures.csv")

print(df_player_fixtures.shape)
print(df_player_fixtures.columns.tolist())

(4230, 15)
['id', 'player', 'position', 'price', 'status', 'squadId', 'team', 'abbr', 'group', 'elo', 'total_tilt', 'tilt_cat', 'round_id', 'opp_abbr', 'opp_elo']


## Elo Model

Source - https://www.sciencedirect.com/science/article/pii/S037722172500935X

In [18]:
def win_expectancy(elo_team, elo_opp):
    """Standard Elo win expectancy formula. Neutral ground."""
    return 1 / (1 + 10 ** ((elo_opp - elo_team) / 400))

def expected_goals(we):
    """
    Quartic polynomial mapping win expectancy -> xG scored, neutral ground.
    Source: football-rankings.info — fitted on ~40,000 NT matches. R²=0.976
    Two-regime model: separate polynomials for We < 0.9 and We >= 0.9.
    """
    if isinstance(we, (int, float)):
        # Scalar case
        if we < 0.9:
            return (3.90388 * we**4 - 0.58486 * we**3
                    - 2.98315 * we**2 + 3.13160 * we + 0.33193)
        else:
            w = we - 0.9
            return (308097.45501 * w**4 - 42803.04696 * w**3
                    + 2116.35304 * w**2 - 9.61869 * w + 2.86899)
    else:
        # Pandas Series / numpy array case
        low = (3.90388 * we**4 - 0.58486 * we**3
               - 2.98315 * we**2 + 3.13160 * we + 0.33193)
        high = (308097.45501 * (we - 0.9)**4 - 42803.04696 * (we - 0.9)**3
                + 2116.35304 * (we - 0.9)**2 - 9.61869 * (we - 0.9) + 2.86899)
        return np.where(we < 0.9, low, high)

def clean_sheet_prob(xg_conceded):
    """Poisson probability of conceding zero goals."""
    return np.exp(-xg_conceded)

# Apply to every fixture row
df_player_fixtures["win_exp"] = win_expectancy(
    df_player_fixtures["elo"],
    df_player_fixtures["opp_elo"]
)
df_player_fixtures["xg_scored"]     = expected_goals(df_player_fixtures["win_exp"])
df_player_fixtures["xg_conceded"]   = expected_goals(1 - df_player_fixtures["win_exp"])
df_player_fixtures["p_clean_sheet"] = clean_sheet_prob(df_player_fixtures["xg_conceded"])

# Sanity check - Spain's fixtures
df_player_fixtures[df_player_fixtures["abbr"] == "ENG"][
    ["opp_abbr", "opp_elo", "win_exp", "xg_scored", "xg_conceded", "p_clean_sheet"]
].drop_duplicates().round(3)

,opp_abbr,opp_elo,win_exp,xg_scored,xg_conceded,p_clean_sheet
1194,CRO,1876.1,0.708,1.826,1.006,0.366
1195,GHA,1678.6,0.883,2.742,0.657,0.518
1196,PAN,1739.1,0.842,2.467,0.752,0.471


## xP per 90 by position

In [19]:
def xpts_per_90(position, xg_scored, xg_conceded, p_clean_sheet):
    """
    Convert match outcome probabilities into expected Fantasy points per 90 mins
    for a single player of the given position.
    
    Goal/assist shares represent one player's slice of team xG/xA,
    divided by typical number of starters per position group.
    
    Typical starters: GK=1, DEF=4, MID=4, FWD=2
    Position group shares: GK=1%, DEF=10%, MID=35%, FWD=54%
    Per-player shares: GK=1%, DEF=2.5%, MID=8.75%, FWD=27%
    """

    # Per-player share of team xG
    goal_share = {
        "GK":  0.01,
        "DEF": 0.025,
        "MID": 0.0875,
        "FWD": 0.27,
    }

    # Per-player share of team assists (shifted toward MID)
    assist_share = {
        "GK":  0.01,
        "DEF": 0.025,
        "MID": 0.1125,
        "FWD": 0.22,
    }

    # Expected goals/assists for this one player
    pos_xg      = xg_scored * goal_share[position]
    pos_assists = xg_scored * assist_share[position]

    # Points components
    pts_goals   = pos_xg * GOAL_PTS[position]
    pts_assists = pos_assists * ASSIST_PTS
    pts_cs      = p_clean_sheet * CLEAN_SHEET_PTS[position]

    # Goals conceded penalty (GK/DEF only)
    if position in ("GK", "DEF"):
        pts_conceded = max(0, xg_conceded - 1) * GOALS_CONCEDED_PTS[position]
    else:
        pts_conceded = 0

    return pts_goals + pts_assists + pts_cs + pts_conceded


# Apply to every row
df_player_fixtures["xpts_p90"] = df_player_fixtures.apply(
    lambda r: xpts_per_90(
        r["position"],
        r["xg_scored"],
        r["xg_conceded"],
        r["p_clean_sheet"]
    ), axis=1
)

# Sanity check - one player per position for Spain
for pos in ["GK", "DEF", "MID", "FWD"]:
    sample = df_player_fixtures[
        (df_player_fixtures["abbr"] == "ESP") &
        (df_player_fixtures["position"] == pos)
    ][["player", "position", "opp_abbr", "xg_scored", "xg_conceded", "p_clean_sheet", "xpts_p90"]].head(3)
    print(sample.to_string())
    print()

          player position opp_abbr  xg_scored  xg_conceded  p_clean_sheet  xpts_p90
2700  David Raya       GK      CPV   3.772639     0.521126       0.593851  3.421973
2701  David Raya       GK      KSA   3.664433     0.529759       0.588747  3.383468
2702  David Raya       GK      URU   1.861178     0.990446       0.371411  2.080397

           player position opp_abbr  xg_scored  xg_conceded  p_clean_sheet  xpts_p90
2670  Pedro Porro      DEF      CPV   3.772639     0.521126       0.593851  3.912416
2671  Pedro Porro      DEF      KSA   3.664433     0.529759       0.588747  3.859844
2672  Pedro Porro      DEF      URU   1.861178     0.990446       0.371411  2.322350

           player position opp_abbr  xg_scored  xg_conceded  p_clean_sheet  xpts_p90
2661  Yéremy Pino      MID      CPV   3.772639     0.521126       0.593851  3.847752
2662  Yéremy Pino      MID      KSA   3.664433     0.529759       0.588747  3.749321
2663  Yéremy Pino      MID      URU   1.861178     0.990446       0

## xMins

In [20]:
#sanity check
xmins = pd.read_csv("data/xmins.csv")
# Show England players
eng_ids = df_player_fixtures[df_player_fixtures["abbr"] == "ENG"]["id"].unique()
xmins[xmins["id"].isin(eng_ids)].sort_values("xmins", ascending=False)

,id,xmins
13,477,90
2,461,88
7,468,88
20,491,85
4,463,85
19,488,85
18,486,85
8,469,80
0,457,80
9,471,77


Merge xMins and compute total projected points

In [21]:
# Load xmins
df_xmins = pd.read_csv("data/xmins.csv")

# Merge onto player fixtures
df_player_fixtures = df_player_fixtures.merge(df_xmins, on="id", how="left")

# Default any missing xmins to 60
df_player_fixtures["xmins"] = df_player_fixtures["xmins"].fillna(60)

# Appearance points based on xmins
def appearance_pts(xmins):
    if xmins == 0:
        return 0
    elif xmins < 60:
        return 1
    else:
        return 2

df_player_fixtures["app_pts"] = df_player_fixtures["xmins"].apply(appearance_pts)

# xpts per game = (xpts_p90 / 90) * xmins + appearance pts
df_player_fixtures["xpts_game"] = (
    (df_player_fixtures["xpts_p90"] / 90) * df_player_fixtures["xmins"]
    + df_player_fixtures["app_pts"]
)

# Total group stage xpts = sum across 3 fixtures
df_projections = (
    df_player_fixtures.groupby(["id", "player", "position", "price", "team", "abbr"])
    .agg(xpts=("xpts_game", "sum"))
    .reset_index()
    .sort_values("xpts", ascending=False)
)

df_projections["xpts"] = df_projections["xpts"].round(2)

# Sanity check — England
df_projections[df_projections["abbr"] == "ENG"].head(15)

,id,player,position,price,team,abbr,xpts
405,468,Harry Kane,FWD,10.5,England,ENG,19.83
400,461,Marc Guéhi,DEF,5.1,England,ENG,14.34
402,463,Ezri Konsa,DEF,4.8,England,ENG,14.06
411,477,Jordan Pickford,GK,4.8,England,ENG,13.61
398,457,Nico O'Reilly,DEF,4.7,England,ENG,13.58
1344,1709,Reece James,DEF,5.2,England,ENG,13.11
416,486,Elliot Anderson,MID,6.5,England,ENG,13.01
417,488,Declan Rice,MID,7.0,England,ENG,13.01
418,491,Jude Bellingham,MID,8.3,England,ENG,13.01
406,469,Bukayo Saka,MID,9.5,England,ENG,12.60
